# 11 — Confronto tra tracker (E2)

**Esperimento 2 della tesi.** Tracking dei giocatori sulle sequenze di **basket di SportsMOT (val)**, che hanno box e identità su tutti i frame → metriche MOT standard (**HOTA, MOTA, IDF1, ID switch**) calcolate con il modulo di valutazione di `trackers` (Roboflow).

| Metodo | Tipo |
|---|---|
| SORT, ByteTrack, OC-SORT, BoT-SORT, C-BIoU | tracking-by-detection (libreria `trackers`) |
| SAM2 inizializzato al frame 0 | baseline della tua pipeline attuale |
| **SAM2 + ri-detection periodica** | miglioria sperimentale 1 (idea del vecchio notebook 06) |
| **+ interpolazione dei buchi** | miglioria sperimentale 2 (post-processing, applicata a tutti) |
| detector COCO vs detector fine-tuned | effetto del dominio sul tracking |

Protocollo: le detection vengono calcolate **una sola volta** per sequenza e salvate; tutti i tracker ricevono **le stesse detection** → il confronto misura solo l'associazione.
Runtime: Colab GPU T4.

In [ ]:
%pip install -q ultralytics trackers supervision

In [ ]:
# ==== SETUP COMUNE (uguale in tutti i notebook) ====
import sys, os
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Tesi_Basket')   # progetto su Drive
    FAST = Path('/content/fast')                                 # disco locale Colab (veloce)
else:
    ROOT = Path(r'F:\Tesi Basket')                               # progetto in locale
    FAST = ROOT / '_cache'
FAST.mkdir(parents=True, exist_ok=True)
assert ROOT.is_dir(), f'Cartella del progetto non trovata: {ROOT}'
print('Progetto:', ROOT)

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')   # riduce la frammentazione della memoria GPU
import json, time, shutil
import numpy as np, pandas as pd, cv2, torch
import supervision as sv
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

EXP = ROOT / 'experiments' / 'E2_tracking'
EXP.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NESSUNA')

## 1. Configurazione

In [ ]:
DETECTORS = {
    'ft':      ROOT / 'models' / 'detector_best.pt',     # scelto nel notebook 10 (RT-DETR-L fine-tuned)
    'yolo11m': ROOT / 'models' / 'yolo11m_ft.pt',        # miglior YOLO: più preciso e 3.5x più veloce
    'coco':    'yolo11s.pt',                              # pre-addestrato COCO (classe person)
}
PLAYER_CLASSES = {'player', 'player-in-possession', 'player-jump-shot',
                  'player-layup-dunk', 'player-shot-block', 'person'}
DET_CONF_MIN = 0.05        # salviamo anche detection deboli: ByteTrack & co. le usano
IMGSZ = 640

RUN_SAM2 = True
SAM2_MAX_SEQ = 3           # SAM2 è lento: lo eseguiamo su un sottoinsieme di sequenze...
SAM2_MAX_FRAMES = 300      # ...e sui primi 300 frame (12 s); il confronto con gli altri tracker usa lo stesso tratto
SAM2_SCALE = 0.667         # frame ridimensionati (1280x720 -> 854x480, come le tue clip NBA) per memoria e velocità
SAM2_MEMORY_KEEP = 16      # frame di memoria conservati (SAM2 ne usa al massimo 7 + 16 puntatori)
MAX_OBJECTS = 12           # al massimo 12 oggetti seguiti da SAM2 (10 giocatori + margine)
SAM2_DETECTOR = 'ft'
REDETECT_EVERY = 5         # frame tra due controlli del detector (variante SAM2+redet)
INTERP_MAX_GAP = 20        # frame massimi da interpolare

## 2. Dati: sequenze di basket SportsMOT

In [ ]:
# ==== SportsMOT: download (val), ricerca sequenze e lista basket ====
import subprocess, urllib.request, configparser

SMOT_DIR = FAST / 'SportsMOT'
SMOT_CACHE = ROOT / 'datasets' / '_cache_sportsmot'     # zip conservati su Drive: niente ri-download
BASKET_LIST = ROOT / 'datasets' / 'sportsmot_basketball.txt'

def sportsmot_val_sequences():
    if not SMOT_DIR.exists() or not any(SMOT_DIR.rglob('gt.txt')):
        SMOT_CACHE.mkdir(parents=True, exist_ok=True)
        subprocess.run(['trackers', 'download', 'sportsmot', '--split', 'val',
                        '--output', str(SMOT_DIR), '--cache-dir', str(SMOT_CACHE)], check=True)
    seqs = sorted({p.parent.parent for p in SMOT_DIR.rglob('gt/gt.txt')})
    assert seqs, f'Nessuna sequenza trovata in {SMOT_DIR}'
    return seqs

def basketball_names():
    if not BASKET_LIST.is_file():
        url = 'https://huggingface.co/datasets/MCG-NJU/SportsMOT/resolve/main/splits_txt/basketball.txt'
        try:
            BASKET_LIST.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(url, BASKET_LIST)
        except Exception as e:
            raise RuntimeError(
                f'Non riesco a scaricare la lista basket ({e}).\n'
                f'Scaricala a mano da {url} (serve login Hugging Face) e salvala in {BASKET_LIST}')
    return set(BASKET_LIST.read_text().split())

def seq_info(seq):
    ini = seq / 'seqinfo.ini'
    fps, n = 25.0, len(list((seq / 'img1').glob('*.jpg')))
    if ini.is_file():
        c = configparser.ConfigParser(); c.read(ini)
        fps = float(c['Sequence'].get('frameRate', fps))
    return fps, n

basket = basketball_names()
SEQS = [s for s in sportsmot_val_sequences() if s.name in basket]
info = pd.DataFrame([{'seq': s.name, 'fps': seq_info(s)[0], 'frames': seq_info(s)[1]} for s in SEQS])
print(len(SEQS), 'sequenze di basket |', info.frames.sum(), 'frame')

# GT in una cartella dedicata (layout MOT: <seq>/gt/gt.txt) con solo le sequenze di basket
GT_DIR = FAST / 'gt_basket'
for s in SEQS:
    d = GT_DIR / s.name / 'gt'; d.mkdir(parents=True, exist_ok=True)
    shutil.copy2(s / 'gt' / 'gt.txt', d / 'gt.txt')
info

## 3. Detection (una volta per sequenza e per detector, salvate su Drive)

In [ ]:
from ultralytics import YOLO, RTDETR
DET_CACHE = EXP / 'detections'

def frames_of(seq):
    return sorted((seq / 'img1').glob('*.jpg'))

def run_detector(det_name, weights):
    model = RTDETR(str(weights)) if 'rtdetr' in str(weights) else YOLO(str(weights))
    keep = [i for i, n in model.names.items() if n in PLAYER_CLASSES]
    out_dir = DET_CACHE / det_name; out_dir.mkdir(parents=True, exist_ok=True)
    times = []
    for seq in tqdm(SEQS, desc=f'detector {det_name}'):
        f = out_dir / f'{seq.name}.npy'
        if f.is_file(): continue
        rows = []
        for fi, p in enumerate(frames_of(seq)):
            t0 = time.perf_counter()
            r = model.predict(str(p), conf=DET_CONF_MIN, iou=0.7, imgsz=IMGSZ, classes=keep,
                              agnostic_nms=True, verbose=False)[0]
            times.append(time.perf_counter() - t0)
            for (x1, y1, x2, y2), c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
                rows.append([fi + 1, x1, y1, x2, y2, c])          # frame 1-based
        np.save(f, np.array(rows, dtype=np.float32).reshape(-1, 6))
    if times: print(f'{det_name}: {1/np.mean(times):.1f} FPS detector')

for name, w in DETECTORS.items():
    run_detector(name, w)

def load_dets(det_name, seq):
    a = np.load(DET_CACHE / det_name / f'{seq.name}.npy')
    return {int(f): a[a[:, 0] == f, 1:] for f in np.unique(a[:, 0])}

## 4. Tracker tracking-by-detection

Parametri **di default** della libreria (nessun tuning sul test: evita di "barare"). Output in formato MOT: `frame,id,x,y,w,h,conf,-1,-1,-1`.

In [ ]:
import importlib
import trackers as T

def make_tracker(name, fps):
    cls = {'SORT': 'SORTTracker', 'ByteTrack': 'ByteTrackTracker', 'OC-SORT': 'OCSORTTracker',
           'BoT-SORT': 'BoTSORTTracker', 'C-BIoU': 'CBIoUTracker'}[name]
    if not hasattr(T, cls):          # fallback se la classe non è esportata al primo livello
        mod = {'SORTTracker': 'sort', 'ByteTrackTracker': 'bytetrack', 'OCSORTTracker': 'ocsort',
               'BoTSORTTracker': 'botsort', 'CBIoUTracker': 'cbiou'}[cls]
        return getattr(importlib.import_module(f'trackers.core.{mod}.tracker'), cls)(frame_rate=fps)
    return getattr(T, cls)(frame_rate=fps)

TRACKERS = ['SORT', 'ByteTrack', 'OC-SORT', 'BoT-SORT', 'C-BIoU']
NEEDS_FRAME = {'BoT-SORT', 'C-BIoU'}       # compensazione movimento camera: serve l'immagine
RES_DIR = EXP / 'tracker_outputs'

def write_mot(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        for fr, tid, x1, y1, x2, y2, c in rows:
            f.write(f'{int(fr)},{int(tid)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{c:.3f},-1,-1,-1\n')

def run_tbd(tracker_name, det_name, seq):
    out = RES_DIR / f'{det_name}__{tracker_name}' / f'{seq.name}.txt'
    if out.is_file(): return None
    fps, n = seq_info(seq)
    dets = load_dets(det_name, seq)
    trk = make_tracker(tracker_name, fps)
    paths = frames_of(seq)
    rows, t_total = [], 0.0
    for fi in range(1, n + 1):
        d = dets.get(fi, np.zeros((0, 5), np.float32))
        detections = sv.Detections(xyxy=d[:, :4].astype(np.float32), confidence=d[:, 4].astype(np.float32),
                                   class_id=np.zeros(len(d), dtype=int))
        frame = cv2.imread(str(paths[fi - 1])) if tracker_name in NEEDS_FRAME else None
        t0 = time.perf_counter()
        tracked = trk.update(detections, frame=frame)
        t_total += time.perf_counter() - t0
        conf = tracked.confidence if tracked.confidence is not None else np.ones(len(tracked))
        for (x1, y1, x2, y2), tid, c in zip(tracked.xyxy, tracked.tracker_id, conf):
            if tid is not None and tid >= 0:
                rows.append((fi, tid + 1, x1, y1, x2, y2, c))
    write_mot(out, rows)
    return n / max(t_total, 1e-9)

speed = []
for det_name in DETECTORS:
    for tr in TRACKERS:
        for seq in tqdm(SEQS, desc=f'{det_name} + {tr}', leave=False):
            fps_tr = run_tbd(tr, det_name, seq)
            if fps_tr: speed.append({'method': f'{det_name}__{tr}', 'seq': seq.name, 'tracker_fps': fps_tr})
if speed:
    pd.DataFrame(speed).to_csv(EXP / f'speed_tbd_{int(time.time())}.csv', index=False)
print('Fatto.')

## 5. SAM2: baseline (frame 0) e variante con ri-detection

- **SAM2-f0**: come la pipeline originale, i prompt sono le detection del primo frame; chi entra dopo non viene mai tracciato.
- **SAM2-redet** (miglioria): ogni `REDETECT_EVERY` frame confrontiamo le detection con le maschere attive; una detection che non si sovrappone a nessuna traccia (IoU < 0.3) per **due controlli consecutivi** diventa un nuovo oggetto. Il predictor viene re-inizializzato con le box correnti (stessi ID) + i nuovi oggetti. Le tracce la cui maschera è vuota da più di 2 controlli vengono chiuse.

Usa il fork `Gy920/segment-anything-2-real-time` allo stesso commit della tua baseline.

In [ ]:
import subprocess, importlib
SAM2_DIR = FAST / 'segment-anything-2-real-time'
SAM2_COMMIT = 'e3623cda3f748df999f76db663f08da7a884a820'
SAM2_CKPT = ROOT / 'models' / 'sam2.1_hiera_large.pt'

if RUN_SAM2:
    if not SAM2_DIR.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Gy920/segment-anything-2-real-time.git', str(SAM2_DIR)], check=True)
        subprocess.run(['git', '-C', str(SAM2_DIR), 'checkout', SAM2_COMMIT], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SAM2_DIR)], check=True)
    if not SAM2_CKPT.is_file():
        SAM2_CKPT.parent.mkdir(exist_ok=True)
        subprocess.run(['wget', '-q', '-O', str(SAM2_CKPT),
                        'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt'], check=True)
    sys.path.insert(0, str(SAM2_DIR))
    for m in [m for m in sys.modules if m == 'sam2' or m.startswith('sam2.')]: del sys.modules[m]
    importlib.invalidate_caches()
    import warnings
    warnings.filterwarnings('ignore', category=UserWarning, module=r'.*sam2.*')   # avviso connected_components: innocuo
    from sam2.build_sam import build_sam2_camera_predictor
    cwd = os.getcwd(); os.chdir(SAM2_DIR)       # la config Hydra viene cercata relativa al repo
    predictor = build_sam2_camera_predictor('configs/sam2.1/sam2.1_hiera_l.yaml', str(SAM2_CKPT), device='cuda')
    os.chdir(cwd)

    # --- Fix out-of-memory su GPU T4 (e altre pre-Ampere) -------------------------------------------
    # Sulla T4 PyTorch non ha un kernel di attenzione efficiente per la memory-attention di SAM2
    # (head dim 256): usa il kernel "math", che costruisce la matrice oggetti x 4096 x ~29k token
    # (≈1.2 GB ciascuna). Con 11-12 giocatori la T4 esaurisce la memoria. Qui calcoliamo la stessa
    # attenzione a blocchi di query: risultato matematicamente identico, picco di memoria ~10 volte più basso.
    # Su L4 / A100 (Ampere+) la patch non si attiva e SAM2 usa Flash Attention.
    import types, torch.nn.functional as _TF
    import sam2.modeling.sam.transformer as _sam2_tr
    _sdpa_orig = _TF.scaled_dot_product_attention
    _OLD_GPU = torch.cuda.get_device_capability(0)[0] < 8
    def _sdpa_chunked(q, k, v, *args, **kw):
        if _OLD_GPU and q.dim() == 4:
            B, H, Lq, _ = q.shape; Lk = k.shape[-2]
            step = max(1, (1 << 26) // max(1, B * H * Lk))          # ~64M elementi per blocco
            if step < Lq:
                return torch.cat([_sdpa_orig(q[:, :, i:i + step], k, v, *args, **kw)
                                  for i in range(0, Lq, step)], dim=2)
        return _sdpa_orig(q, k, v, *args, **kw)
    _F_proxy = types.SimpleNamespace(**{n: getattr(_TF, n) for n in dir(_TF) if not n.startswith('__')})
    _F_proxy.scaled_dot_product_attention = _sdpa_chunked
    _sam2_tr.F = _F_proxy
    print('GPU:', torch.cuda.get_device_name(0), '| attenzione a blocchi attiva:', _OLD_GPU)
    print('SAM2 pronto.')

In [ ]:
def box_iou(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = lambda r: (r[:, 2] - r[:, 0]) * (r[:, 3] - r[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)

def masks_to_boxes(obj_ids, logits):
    masks = (logits > 0).cpu().numpy()
    if masks.ndim == 4: masks = masks[:, 0]
    out = {}
    for oid, m in zip(obj_ids, masks):
        if m.any():
            m = sv.filter_segments_by_distance(m, relative_distance=0.03, mode='edge').astype(bool)
            ys, xs = np.where(m)
            if len(xs): out[int(oid)] = np.array([xs.min(), ys.min(), xs.max(), ys.max()], np.float32)
    return out

def sam2_init(rgb, boxes_by_id):
    if hasattr(predictor, 'reset_state'):
        try: predictor.reset_state()
        except Exception: pass
    predictor.frame_idx = 0
    predictor.load_first_frame(rgb)
    ids, logits = [], None
    for oid, b in boxes_by_id.items():
        _, ids, logits = predictor.add_new_prompt(frame_idx=0, obj_id=int(oid), bbox=b.reshape(1, 4))
    return ids, logits

def prune_memory(keep=SAM2_MEMORY_KEEP):
    # Il predictor real-time conserva gli output di TUTTI i frame: teniamo solo gli ultimi `keep` (evita l'out-of-memory).
    st = getattr(predictor, 'condition_state', None)
    if not isinstance(st, dict): return
    lim = predictor.frame_idx - keep
    dicts = [st.get('output_dict', {})] + list(st.get('output_dict_per_obj', {}).values())
    for od in dicts:
        d = od.get('non_cond_frame_outputs', {}) if isinstance(od, dict) else {}
        for k in [k for k in list(d) if isinstance(k, int) and k < lim]:
            del d[k]

def run_sam2(seq, det_name, redetect, conf_init=0.4, conf_new=0.5):
    name = 'SAM2-redet' if redetect else 'SAM2-f0'
    out = RES_DIR / f'{det_name}__{name}' / f'{seq.name}.txt'
    if out.is_file():
        old = np.loadtxt(out, delimiter=',', ndmin=2)
        if old.size == 0 or old[:, 0].max() <= SAM2_MAX_FRAMES: return None
        out.unlink()                                           # file di una run precedente con impostazioni diverse
        stale = RES_DIR / f'{det_name}__{name}+interp' / f'{seq.name}.txt'
        if stale.is_file(): stale.unlink()
    dets = load_dets(det_name, seq)
    paths = frames_of(seq)[:SAM2_MAX_FRAMES]
    S = SAM2_SCALE
    d0 = dets.get(1, np.zeros((0, 5)))
    d0 = d0[d0[:, 4] >= conf_init]
    if len(d0) == 0: write_mot(out, []); return None
    boxes = {i + 1: b[:4] * S for i, b in enumerate(d0)}          # prompt nelle coordinate ridimensionate
    next_id, pending, missing = len(boxes) + 1, np.zeros((0, 4)), {}
    rows, t0 = [], time.perf_counter()
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16):
        for fi, p in enumerate(tqdm(paths, desc=f'{name} {seq.name}', leave=False), start=1):
            rgb = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
            if S != 1: rgb = cv2.resize(rgb, None, fx=S, fy=S, interpolation=cv2.INTER_AREA)
            if fi == 1:
                ids, logits = sam2_init(rgb, boxes)
            else:
                ids, logits = predictor.track(rgb)
            prune_memory()
            cur_s = masks_to_boxes(ids, logits)                  # box nelle coordinate ridimensionate
            cur = {oid: b / S for oid, b in cur_s.items()}       # box nelle coordinate originali (per GT e detection)
            for oid, b in cur.items():
                rows.append((fi, oid, *b, 1.0))
            if redetect and fi > 1 and fi % REDETECT_EVERY == 0:
                for oid in [int(i) for i in ids]:
                    missing[oid] = 0 if oid in cur else missing.get(oid, 0) + 1
                d = dets.get(fi, np.zeros((0, 5))); d = d[d[:, 4] >= conf_new][:, :4]
                cur_boxes = np.array(list(cur.values())).reshape(-1, 4)
                cand = d[box_iou(d, cur_boxes).max(axis=1, initial=0) < 0.3] if len(d) else np.zeros((0, 4))
                confirmed = cand[box_iou(cand, pending).max(axis=1, initial=0) > 0.5] if len(cand) else cand
                pending = cand
                alive = {oid: b for oid, b in cur_s.items() if missing.get(oid, 0) <= 2}
                room = MAX_OBJECTS - len(alive)                  # tetto agli oggetti: evita l'esplosione (e l'out-of-memory)
                if len(confirmed) and room > 0:
                    for b in confirmed[:room]:
                        alive[next_id] = b * S; next_id += 1
                    sam2_init(rgb, alive)
                    pending = np.zeros((0, 4))
    write_mot(out, rows)
    print(f'{name} {seq.name}: picco memoria GPU {torch.cuda.max_memory_allocated()/2**30:.1f} GB')
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    return len(paths) / (time.perf_counter() - t0)

if RUN_SAM2:
    sam_seqs = SEQS[:SAM2_MAX_SEQ]
    sam_speed = []
    for seq in sam_seqs:
        for redet in (False, True):
            f = run_sam2(seq, SAM2_DETECTOR, redet)
            if f: sam_speed.append({'method': f"{SAM2_DETECTOR}__{'SAM2-redet' if redet else 'SAM2-f0'}", 'seq': seq.name, 'tracker_fps': f})
    if sam_speed: pd.DataFrame(sam_speed).to_csv(EXP / f'speed_sam2_{int(time.time())}.csv', index=False)

## 6. Miglioria: interpolazione dei buchi nelle tracce

Per ogni ID, se una traccia sparisce per ≤ `INTERP_MAX_GAP` frame e poi riappare, le box mancanti vengono interpolate linearmente (post-processing offline, come GSI/linear interpolation in StrongSORT). Si applica a **tutti** i metodi → variante `+interp`.

In [ ]:
def interpolate_file(src, dst, max_gap=INTERP_MAX_GAP):
    a = np.loadtxt(src, delimiter=',', ndmin=2)
    if a.size == 0: dst.parent.mkdir(parents=True, exist_ok=True); dst.write_text(''); return
    out = [a]
    for tid in np.unique(a[:, 1]):
        t = a[a[:, 1] == tid]; t = t[np.argsort(t[:, 0])]
        for r0, r1 in zip(t[:-1], t[1:]):
            gap = int(r1[0] - r0[0])
            if 1 < gap <= max_gap + 1:
                for k in range(1, gap):
                    w = k / gap; r = r0.copy(); r[0] = r0[0] + k
                    r[2:6] = (1 - w) * r0[2:6] + w * r1[2:6]; r[6] = min(r0[6], r1[6])
                    out.append(r[None])
    a = np.concatenate(out); a = a[np.lexsort((a[:, 1], a[:, 0]))]
    dst.parent.mkdir(parents=True, exist_ok=True)
    np.savetxt(dst, a, delimiter=',', fmt=['%d', '%d', '%.2f', '%.2f', '%.2f', '%.2f', '%.3f', '%d', '%d', '%d'])

for method_dir in sorted(RES_DIR.iterdir()):
    if method_dir.name.endswith('+interp'): continue
    for f in method_dir.glob('*.txt'):
        dst = RES_DIR / (method_dir.name + '+interp') / f.name
        if not dst.is_file(): interpolate_file(f, dst)
print(sorted(p.name for p in RES_DIR.iterdir()))

## 7. Valutazione (HOTA, MOTA, IDF1, ID switch)

Due tabelle:
1. **tutte le sequenze di basket** — tracker tracking-by-detection;
2. **sottoinsieme SAM2** — tutti i metodi, compresi SAM2, sulle stesse sequenze (confronto equo).

In [ ]:
try:
    from trackers.eval import evaluate_mot_sequences
except ImportError:
    from trackers.eval.evaluate import evaluate_mot_sequences

def copy_upto(src, dst, max_frame):
    # Copia un file MOT tenendo solo i frame <= max_frame (None = tutti).
    lines = Path(src).read_text().splitlines()
    if max_frame: lines = [l for l in lines if l.strip() and int(float(l.split(',')[0])) <= max_frame]
    Path(dst).write_text('\n'.join(lines) + ('\n' if lines else ''))

def evaluate(method, seq_names, max_frame=None):
    tag = '_'.join(sorted(seq_names))[:60] + f'_{len(seq_names)}' + (f'_f{max_frame}' if max_frame else '')
    gt_sub = FAST / 'gt_eval' / tag
    trk_sub = FAST / 'trk_eval' / method / tag
    for s in seq_names:
        g = gt_sub / s / 'gt'; g.mkdir(parents=True, exist_ok=True)
        if not (g / 'gt.txt').exists(): copy_upto(GT_DIR / s / 'gt' / 'gt.txt', g / 'gt.txt', max_frame)
        trk_sub.mkdir(parents=True, exist_ok=True)
        src = RES_DIR / method / f'{s}.txt'
        if not src.is_file(): return None
        copy_upto(src, trk_sub / f'{s}.txt', max_frame)
    r = evaluate_mot_sequences(gt_dir=gt_sub, tracker_dir=trk_sub, metrics=['CLEAR', 'HOTA', 'Identity'])
    agg = r.aggregate
    return {'method': method, 'HOTA': agg.HOTA.HOTA, 'DetA': getattr(agg.HOTA, 'DetA', np.nan),
            'AssA': getattr(agg.HOTA, 'AssA', np.nan), 'MOTA': agg.CLEAR.MOTA, 'IDF1': agg.Identity.IDF1,
            'IDSW': agg.CLEAR.IDSW}, r

methods = sorted(p.name for p in RES_DIR.iterdir() if p.is_dir())
all_names = [s.name for s in SEQS]
rows, PER_SEQ = [], {}
for m in methods:
    if 'SAM2' in m: continue
    out = evaluate(m, all_names)
    if out: rows.append(out[0]); PER_SEQ[m] = out[1]
res_all = pd.DataFrame(rows).set_index('method').sort_values('HOTA', ascending=False).round(4)
res_all.to_csv(EXP / 'results_all_basket.csv')
display(res_all)

if RUN_SAM2:
    sam_names = [s.name for s in SEQS[:SAM2_MAX_SEQ]]
    rows = [o[0] for m in methods if (o := evaluate(m, sam_names, SAM2_MAX_FRAMES))]
    res_sub = pd.DataFrame(rows).set_index('method').sort_values('HOTA', ascending=False).round(4)
    res_sub.to_csv(EXP / 'results_sam2_subset.csv')
    display(res_sub)

## 8. Grafici per la tesi

In [ ]:
FIG = EXP / 'figures'; FIG.mkdir(exist_ok=True)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})

def bars(df, title, fname):
    d = df[['HOTA', 'MOTA', 'IDF1']]
    fig, ax = plt.subplots(figsize=(max(8, 0.55 * len(d)), 4.5))
    x = np.arange(len(d)); w = 0.27
    for i, (col, color) in enumerate(zip(d.columns, ['#2a6f97', '#e07a5f', '#81b29a'])):
        ax.bar(x + (i - 1) * w, d[col], w, label=col, color=color)
    ax.set_xticks(x, d.index, rotation=35, ha='right'); ax.set_ylim(0, 1); ax.legend(ncol=3)
    ax.set_title(title); fig.tight_layout(); fig.savefig(FIG / fname, dpi=200); plt.show()

bars(res_all, 'Tracking giocatori — SportsMOT basket (val)', 'tracker_metriche.png')
if RUN_SAM2: bars(res_sub, f'Confronto con SAM2 — {SAM2_MAX_SEQ} sequenze, primi {SAM2_MAX_FRAMES} frame', 'tracker_metriche_sam2.png')

# Effetto dell'interpolazione: delta per tracker
base = res_all[~res_all.index.str.endswith('+interp')]
delta = pd.DataFrame({m: res_all.loc[m + '+interp', ['HOTA', 'MOTA', 'IDF1']] - base.loc[m, ['HOTA', 'MOTA', 'IDF1']]
                      for m in base.index if m + '+interp' in res_all.index}).T
delta.to_csv(EXP / 'delta_interpolazione.csv'); display(delta.round(4))
ax = delta.plot.bar(figsize=(9, 4), color=['#2a6f97', '#e07a5f', '#81b29a'])
ax.axhline(0, color='k', lw=.8); ax.set_title('Guadagno dovuto all\'interpolazione'); plt.tight_layout()
plt.savefig(FIG / 'delta_interpolazione.png', dpi=200); plt.show()

# Detector COCO vs fine-tuned (stesso tracker)
pairs = [(m, m.replace('coco__', 'ft__')) for m in res_all.index if m.startswith('coco__') and m.replace('coco__', 'ft__') in res_all.index]
if pairs:
    dd = pd.DataFrame({b.split('__')[1]: {'COCO': res_all.loc[a, 'HOTA'], 'fine-tuned': res_all.loc[b, 'HOTA']} for a, b in pairs}).T
    ax = dd.plot.bar(figsize=(8, 4), color=['#bbbbbb', '#2a6f97']); ax.set_ylabel('HOTA'); ax.set_ylim(0, 1)
    ax.set_title('Effetto del fine-tuning del detector sul tracking'); plt.tight_layout()
    plt.savefig(FIG / 'detector_effect_tracking.png', dpi=200); plt.show()

## 9. Video qualitativo (una sequenza, due metodi affiancati)

In [ ]:
def render(method, seq, out_path, max_frames=300):
    a = np.loadtxt(RES_DIR / method / f'{seq.name}.txt', delimiter=',', ndmin=2)
    paths = frames_of(seq)[:max_frames]
    fps, _ = seq_info(seq)
    h, w = cv2.imread(str(paths[0])).shape[:2]
    vw = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for fi, p in enumerate(paths, start=1):
        img = cv2.imread(str(p))
        for r in a[a[:, 0] == fi]:
            tid = int(r[1]); x, y, bw, bh = map(int, r[2:6])
            col = tuple(int(c) for c in np.random.default_rng(tid).integers(60, 255, 3))
            cv2.rectangle(img, (x, y), (x + bw, y + bh), col, 2)
            cv2.putText(img, str(tid), (x, max(15, y - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, col, 2)
        cv2.putText(img, method, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        vw.write(img)
    vw.release()

seq = SEQS[0]
for m in ['ft__ByteTrack', 'ft__SAM2-redet']:
    if (RES_DIR / m / f'{seq.name}.txt').is_file():
        render(m, seq, EXP / f'video_{m}_{seq.name}.mp4'); print('Salvato video', m)